# Reproducir la gráfica de demanda de energía industrial (INEN)

Esta libreta corre SISEPUEDE con el input ajustado `sisepuede_raw_inputs_LBY_apr_with_gfr.csv` y las 3 estrategias (BAU, Unconditional, Conditional), y reproduce la gráfica de demanda de energía industrial en PJ por subsector industrial, 2023‑2050.

**Contexto de la lógica del modelo INEN (SISEPUEDE).** La demanda de energía industrial se calcula como:

$$\text{demand}_{c,t} = \bigl(\text{prod}_{c,t}\cdot I^{prod}_{c,0} + \text{gdp}_t\cdot I^{gdp}_{c,0}\bigr)\;\times\; \text{scalar}_{c,t}$$

donde las intensidades iniciales $I^{prod}_{c,0}$ y $I^{gdp}_{c,0}$ vienen de `consumpinit_inen_energy_tj_per_tonne_production_*` y `consumpinit_inen_energy_tj_per_mmm_gdp_*`. El `scalar_inen_energy_demand_*` es el lever de eficiencia.

**Ajuste aplicado al input.** Las intensidades iniciales se escalaron por `0.391` para que la demanda total a 2023 sea ≈63 PJ (rango objetivo 60‑65 PJ). Se eligió escalar las intensidades y no los scalars porque las transformaciones `TX:INEN:INC_EFFICIENCY_PRODUCTION_*` sobrescriben el scalar vía `magnitude_type: final_value`.

In [ ]:
import logging
import os
import pathlib
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

HERE = pathlib.Path(os.getcwd())
SSP_MODELING_DIR = HERE.parent
DATA_DIR = SSP_MODELING_DIR / 'input_data'
OUT_DIR = SSP_MODELING_DIR / 'ssp_run_output'
TRF_DIR = SSP_MODELING_DIR / 'transformations'
sys.path.insert(0, str(HERE))

from utils.logger_utils import setup_clean_logger, mute_external_loggers
logger = setup_clean_logger('nb', logging.INFO)
mute_external_loggers(['sisepuede'])

## 1. Correr SISEPUEDE con las 3 estrategias

Usa exactamente el mismo input CSV que producción (`sisepuede_raw_inputs_LBY_apr_with_gfr.csv`) y las estrategias `BAU=6003`, `Unconditional=6004`, `Conditional=6005`. Si el run ya existe en `ssp_run_output/`, lo reusa; si no, corre el modelo.

Para generar el run: desde la terminal,
```
python run_energy_validation.py --strategies 0,6003,6004,6005
```
(sin `--electricity` para saltarse NemoMod, que en este contenedor no está disponible).

In [ ]:
# Find most recent run
runs = sorted([p for p in OUT_DIR.glob('sisepuede_results_sisepuede_run_*') if p.is_dir()])
assert len(runs) > 0, 'No runs found. Corre primero `python run_energy_validation.py`'
LATEST_RUN = runs[-1]
print('Using run:', LATEST_RUN)

att_primary = pd.read_csv(OUT_DIR / 'ATTRIBUTE_PRIMARY.csv')
att_strategy = pd.read_csv(OUT_DIR / 'ATTRIBUTE_STRATEGY.csv')
mo_path = next(LATEST_RUN.glob('MODEL_OUTPUT_*.csv'))
df_out = pd.read_csv(mo_path)
print('df_out shape:', df_out.shape)

In [ ]:
# MODEL_OUTPUT already contains strategy_id / strategy_code / year (added by run_energy_validation.py)
df = df_out.copy()
if 'strategy_code' not in df.columns:
    df = df.merge(att_primary[['primary_id', 'strategy_id']], on='primary_id', how='left')
    df = df.merge(att_strategy[['strategy_id', 'strategy_code']], on='strategy_id', how='left')
if 'year' not in df.columns:
    df['year'] = df['time_period'].apply(lambda t: 2015 + int(t))

# Restrict to BAU/Unconditional/Conditional and year >= 2023
scenarios_in_chart = {'PFLO:BAU': 'BAU', 'PFLO:UNCONDITIONAL': 'Unconditional', 'PFLO:CONDITIONAL': 'Conditional'}
df_plot = df[df['strategy_code'].isin(scenarios_in_chart.keys())].copy()
df_plot['scenario'] = df_plot['strategy_code'].map(scenarios_in_chart)
df_plot = df_plot[df_plot['year'] >= 2023]
print(df_plot[['year', 'scenario']].drop_duplicates().groupby('scenario').size())

## 2. Preparar la demanda INEN por subsector industrial

El modelo exporta `energy_demand_inen_<categoría>` (en PJ, unidades de config). Se seleccionan las categorías que aparecen en la gráfica objetivo.

In [ ]:
# Categorías del chart, en el orden del legend
chart_cats_order = [
    'cement', 'chemicals', 'electronics', 'glass', 'lime_and_carbonite',
    'metals', 'mining', 'other_product_manufacturing', 'paper', 'plastic',
    'textiles', 'wood',
]

label_map = {
    'cement': 'cement',
    'chemicals': 'chemicals',
    'electronics': 'electronics',
    'glass': 'glass',
    'lime_and_carbonite': 'lime & carbonite',
    'metals': 'metals',
    'mining': 'mining',
    'other_product_manufacturing': 'other manufacturing',
    'paper': 'paper',
    'plastic': 'plastic',
    'textiles': 'textiles',
    'wood': 'wood',
}

# Color map copying the original chart
colors = {
    'cement':                      '#4C5B8B',
    'chemicals':                   '#A6C8E1',
    'electronics':                 '#E69A3E',
    'glass':                       '#F2C28B',
    'lime_and_carbonite':          '#3D8C56',
    'metals':                      '#9BC995',
    'mining':                      '#A39253',
    'other_product_manufacturing': '#F2D774',
    'paper':                       '#5B9896',
    'plastic':                     '#A6CFCA',
    'textiles':                    '#C95C5B',
    'wood':                        '#8A7C6E',
}

cat_cols = [f'energy_demand_inen_{c}' for c in chart_cats_order]
missing = [c for c in cat_cols if c not in df_plot.columns]
assert not missing, f'Missing output columns: {missing}'

In [ ]:
# Pivot per scenario to (year, category) matrix
pivots = {}
for scen in ['BAU', 'Unconditional', 'Conditional']:
    sub = df_plot[df_plot['scenario']==scen].sort_values('year')
    m = sub[['year']+cat_cols].set_index('year')
    m.columns = chart_cats_order
    pivots[scen] = m

# Show totals
for scen, m in pivots.items():
    total = m.sum(axis=1)
    print(f'{scen}: 2023={total.loc[2023]:.2f} PJ, 2050={total.loc[2050]:.2f} PJ')

## 3. Gráfica con 3 paneles (BAU / Unconditional / Conditional)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

for ax, (scen_name, m) in zip(axes, pivots.items()):
    years = m.index.values
    values = [m[c].values for c in chart_cats_order]
    ax.stackplot(
        years, *values,
        labels=[label_map[c] for c in chart_cats_order],
        colors=[colors[c] for c in chart_cats_order],
        alpha=0.85,
    )
    ax.set_title(scen_name, fontsize=13)
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(2023, 2050)
    ax.set_ylim(0, 120)

axes[0].set_ylabel('Peta Joules', fontsize=12)
axes[-1].legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=10, frameon=False)
plt.suptitle('Demanda de energía industrial (INEN) — Libia', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'inen_energy_demand_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Diagnóstico — verificación del ajuste

Confirmar que 2023 cae dentro del objetivo (60‑65 PJ) y que la trayectoria hasta 2050 es consistente con la gráfica referencia.

In [ ]:
for scen, m in pivots.items():
    total = m.sum(axis=1)
    print(f'--- {scen} ---')
    for y in [2023, 2025, 2030, 2035, 2040, 2045, 2050]:
        print(f'  {y}: {total.loc[y]:6.2f} PJ')